# Duka Electronics: Sales Exploration (January to June 2026)

**Analyst:** *Leah Atieno J.* | **Cohort:** *1MillionDevs * | **Date:** *24-11-26*

Engagement 1 of Assignment 3. Data: `duka_sales.csv`, 1,000 orders, generated by `generate_data.py` (seed 2026).

## Executive Summary

- Duka Electronics took **KES 71,792,500** from **1,000 orders** between 1 January and 29 June 2026, an average of **KES 71,792** per order.
- **Nairobi** is the leading branch with **KES 26,106,100** (36.4% of revenue, 376 orders), and the branch ranking is the same by revenue and by order count.
- **Laptops** bring in **57.9%** of revenue (KES 41,535,000) while **Tablets** sell the most units (691 against 639 for laptops).
- Weekends account for **277 orders (27.7%)** and the average order is KES 71,972 against KES 71,724 on weekdays, a gap of only 0.3%.
- **April 2026** was the best month (KES 13,518,300) and **June 2026** the worst (KES 10,903,400), a **24.0%** difference, and the 83 Premium online orders from Nairobi and Mombasa are worth **KES 10,218,000** (14.2% of revenue).

## 1. Setup and loading

Dates are parsed with `parse_dates` so `order_date` is a real datetime rather than text.

In [ ]:
import os, subprocess, sys

# The raw CSVs are not committed. Create them with generate_data.py if they are missing.
RAW_FILES = ["duka_sales.csv"]
if not all(os.path.exists(f) for f in RAW_FILES):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


## 2. First-five-minutes inspection (Task 1)

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
print(df.dtypes)

Shape (rows, columns): (1000, 7)

order_id               int64
order_date    datetime64[us]
branch                   str
product                  str
quantity               int64
channel                  str
unit_price             int64
dtype: object


In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate order_id values:", df["order_id"].duplicated().sum())
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date(),
      f"({(df['order_date'].max() - df['order_date'].min()).days + 1} calendar days)")

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate order_id values: 0
Date range: 2026-01-01 to 2026-06-29 (180 calendar days)


In [ ]:
df[["quantity", "unit_price"]].describe()

,quantity,unit_price
count,"1,000.00","1,000.00"
mean,3.01,"23,918.30"
std,1.42,"23,908.79"
min,1.00,"1,200.00"
25%,2.00,"2,500.00"
50%,3.00,"15,000.00"
75%,4.00,"28,000.00"
max,5.00,"65,000.00"


In [ ]:
for col in ["branch", "product", "channel"]:
    print(df[col].value_counts().to_string(), "\n")
print("Distinct unit_price per product (should all be 1):")
print(df.groupby("product")["unit_price"].nunique().to_string())

branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117 

product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184 

channel
In-store       579
Online         320
Phone order    101 

Distinct unit_price per product (should all be 1):
product
Charger       1
Headphones    1
Laptop        1
Phone         1
Tablet        1


**What the inspection shows**

- **Shape and types:** 1,000 rows and 7 columns. `order_date` is `datetime64`; `quantity` and `unit_price` are integers; `branch`, `product` and `channel` are text categories.
- **Missing values and duplicates:** none in any column, and `order_id` is unique, so every row is one distinct order.
- **Date range:** 1 January to 29 June 2026, which covers the half-year (the last order falls on 29 June, so June has 29 days of data).
- **Categories:** 5 branches (Nairobi has the most orders), 5 products (each has exactly one `unit_price`, so price is a clean product attribute), and 3 channels (In-store is the largest, then Online, then Phone order).
- **Quantity:** between 1 and 5 units per order, mean about 3.
- **Not in the file yet:** there is no `revenue` column, so it has to be created before any money question can be answered.

## 3. Derived columns (Task 2)

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.to_period("M")
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5          # Saturday = 5, Sunday = 6
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df[["order_id", "order_date", "quantity", "unit_price", "revenue",
    "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,order_date,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,3,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,4,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,5,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,5,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,3,2500,7500,2026-03,Saturday,True,Value


In [ ]:
print("Total revenue (KES):", f"{df['revenue'].sum():,}")
print(df.groupby("price_band")["product"].unique().to_string())

Total revenue (KES): 71,792,500
price_band
Premium                [Laptop, Tablet]
Value      [Phone, Headphones, Charger]


`price_band` follows the brief: **Premium** when `unit_price` is KES 28,000 or more (Tablet and Laptop), **Value** otherwise (Phone, Headphones, Charger). A Tablet at exactly 28,000 is Premium because the rule uses `>=`.

## 4. Task 3: Branch rankings by revenue and by orders

In [ ]:
branch = (df.groupby("branch")
            .agg(revenue=("revenue", "sum"), orders=("order_id", "count"),
                 avg_order=("revenue", "mean"))
            .assign(revenue_share_pct=lambda t: t["revenue"] / t["revenue"].sum() * 100))
branch["rank_revenue"] = branch["revenue"].rank(ascending=False).astype(int)
branch["rank_orders"] = branch["orders"].rank(ascending=False).astype(int)
branch = branch.sort_values("revenue", ascending=False)
branch

,revenue,orders,avg_order,revenue_share_pct,rank_revenue,rank_orders
branch,,,,,,
Nairobi,26106100,376,"69,431.12",36.36,1,1
Mombasa,14003700,186,"75,288.71",19.51,2,2
Nakuru,11970400,162,"73,891.36",16.67,3,3
Kisumu,10868100,159,"68,352.83",15.14,4,4
Eldoret,8844200,117,"75,591.45",12.32,5,5


In [ ]:
print("Same ranking by revenue and orders?", (branch["rank_revenue"] == branch["rank_orders"]).all())

Same ranking by revenue and orders? True


**Answer.** Yes, the two rankings are identical: Nairobi (376 orders, KES 26,106,100) > Mombasa (186 orders, KES 14,003,700) > Nakuru (162 orders, KES 11,970,400) > Kisumu (159 orders, KES 10,868,100) > Eldoret (117 orders, KES 8,844,200).

**Why.** Order volume drives revenue here. Branch revenue is orders multiplied by average order value, and the average order value is similar everywhere (KES 68,353 to KES 75,591), while order counts differ a lot (Nairobi has 376 orders against 117 for Eldoret). The gaps in orders are therefore far larger than the gaps in order value and set the order of the table.

**Context and caution.** Nairobi alone brings in 36.4% of revenue. The race for third place is tight on orders (162 for Nakuru and 159 for Kisumu) and Nakuru stays ahead only because its average order is higher (KES 73,891 against KES 68,353). Eldoret has the highest average order of any branch (KES 75,591) but the fewest orders, so growing its order count is its main lever.

## 5. Task 4: Top three products by revenue and by units

In [ ]:
product = (df.groupby("product")
             .agg(revenue=("revenue", "sum"), units=("quantity", "sum"), orders=("order_id", "count"),
                  unit_price=("unit_price", "first"))
             .assign(revenue_share_pct=lambda t: t["revenue"] / t["revenue"].sum() * 100))
top_rev = product.nlargest(3, "revenue")
top_units = product.nlargest(3, "units")
print("Top 3 by revenue:")
print(top_rev[["revenue", "revenue_share_pct", "units"]].to_string())
print()
print("Top 3 by units sold:")
print(top_units[["units", "revenue", "unit_price"]].to_string())

Top 3 by revenue:
          revenue  revenue_share_pct  units
product                                    
Laptop   41535000              57.85    639
Tablet   19348000              26.95    691
Phone     8880000              12.37    592

Top 3 by units sold:
         units   revenue  unit_price
product                             
Tablet     691  19348000       28000
Laptop     639  41535000       65000
Phone      592   8880000       15000


In [ ]:
product.sort_values("revenue", ascending=False)

,revenue,units,orders,unit_price,revenue_share_pct
product,,,,,
Laptop,41535000,639,220,65000,57.85
Tablet,19348000,691,215,28000,26.95
Phone,8880000,592,194,15000,12.37
Headphones,1387500,555,187,2500,1.93
Charger,642000,535,184,1200,0.89


**Answer.** Top three by revenue: **Laptop, Tablet, Phone**. Top three by units: **Tablet, Laptop, Phone**.

**Difference.** Both lists contain the same three products, but the order of the first two changes. Tablets sell the most units (691) but Laptops are first by revenue (KES 41,535,000, 57.9% of the total) because a Laptop costs KES 65,000 against KES 28,000 for a Tablet, so 639 laptops outearn 691 tablets by about 2.1 times.

**Business meaning.** Unit counts for all five products are close (535 to 691), so what separates the products is price, not demand. Headphones and Chargers sell 555 and 535 units but together make only 2.8% of revenue. Stock and promotion effort should follow revenue: a stock-out on Laptops costs far more than one on Chargers.

## 6. Task 5: Weekend and weekday orders

In [ ]:
wk = (df.groupby("is_weekend")
        .agg(orders=("order_id", "count"), total_revenue=("revenue", "sum"),
             avg_order_revenue=("revenue", "mean"))
        .rename(index={False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"}))
wk["order_share_pct"] = wk["orders"] / wk["orders"].sum() * 100
wk["revenue_share_pct"] = wk["total_revenue"] / wk["total_revenue"].sum() * 100

all_days = pd.date_range(df["order_date"].min(), df["order_date"].max())
n_weekend_days = int((all_days.dayofweek >= 5).sum())
n_weekday_days = len(all_days) - n_weekend_days
wk["calendar_days"] = [n_weekday_days, n_weekend_days]
wk["orders_per_day"] = wk["orders"] / wk["calendar_days"]
wk

,orders,total_revenue,avg_order_revenue,order_share_pct,revenue_share_pct,calendar_days,orders_per_day
is_weekend,,,,,,,
Weekday (Mon-Fri),723,51856200,"71,723.65",72.30,72.23,128,5.65
Weekend (Sat-Sun),277,19936300,"71,972.20",27.70,27.77,52,5.33


**Answer.**

| | Orders | Total revenue (KES) | Average order revenue (KES) |
|---|---|---|---|
| Weekday | 723 | 51,856,200 | 71,724 |
| Weekend | 277 | 19,936,300 | 71,972 |

Weekdays hold most orders and revenue simply because there are five weekdays for every two weekend days. Fairer comparisons are the average order and orders per day: the average order is almost identical (weekend KES 71,972 against weekday KES 71,724, a 0.3% gap), and orders per day are 5.33 at weekends against 5.65 on weekdays.

**Interpretation.** Weekend customers do not spend more or less per order, and the order rate per day is also close (5.65 against 5.33, so weekdays are slightly busier per day). There is no evidence here for weekend-only pricing or staffing changes. A difference of this size across 1,000 random-looking orders should not be treated as a pattern.

## 7. Task 6: Best and worst month by revenue

In [ ]:
monthly = df.groupby("month").agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
monthly["days_of_data"] = df.groupby("month")["order_date"].agg(lambda s: s.dt.normalize().nunique())
monthly["revenue_share_pct"] = monthly["revenue"] / monthly["revenue"].sum() * 100
monthly

,revenue,orders,days_of_data,revenue_share_pct
month,,,,
2026-01,11786600,173,31,16.42
2026-02,11772000,159,28,16.40
2026-03,12396800,186,31,17.27
2026-04,13518300,167,30,18.83
2026-05,11415400,153,31,15.90
2026-06,10903400,162,28,15.19


In [ ]:
best_month, worst_month = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
best_rev, worst_rev = monthly["revenue"].max(), monthly["revenue"].min()

print(f"Best month : {best_month}  KES {best_rev:,}")
print(f"Worst month: {worst_month}  KES {worst_rev:,}")
print(f"Best is {(best_rev - worst_rev) / worst_rev * 100:.1f}% higher than worst")
print(f"Worst is {(worst_rev - best_rev) / best_rev * 100:.1f}% lower than best")

Best month : 2026-04  KES 13,518,300
Worst month: 2026-06  KES 10,903,400
Best is 24.0% higher than worst
Worst is -19.3% lower than best


**Answer.** The best month is **April 2026** with KES 13,518,300; the worst is **June 2026** with KES 10,903,400. April is **24.0% higher** than June (equivalently, June is 19.3% lower than April).

**Context.** April stands out against a flat start to the year (January and February were both near KES 11.8 million), and revenue then fell in May and June to below where it began, so the half-year ends on a weak note. June has 29 days of data in this extract (and February has 28), so part of the June gap is calendar length, but one missing day (about 3%) cannot explain a 24% difference. The sensible next step is to ask what changed in April (a promotion, a stock delivery) and why May and June fell.

## 8. Task 7: Premium online orders from Nairobi and Mombasa

In [ ]:
mask = (df["price_band"] == "Premium") & (df["channel"] == "Online") & (df["branch"].isin(["Nairobi", "Mombasa"]))
premium_online = df.loc[mask].copy()

n_pq = len(premium_online)
rev_pq = premium_online["revenue"].sum()
print(f"Orders: {n_pq}")
print(f"Total revenue: KES {rev_pq:,}")
print(f"Share of all orders: {n_pq / len(df) * 100:.1f}%   Share of all revenue: {rev_pq / df['revenue'].sum() * 100:.1f}%")
print(premium_online.groupby(["branch", "product"]).agg(orders=("order_id", "count"), revenue=("revenue", "sum")).to_string())

Orders: 83
Total revenue: KES 10,218,000
Share of all orders: 8.3%   Share of all revenue: 14.2%
                 orders  revenue
branch  product                 
Mombasa Laptop       10  1820000
        Tablet       23  1876000
Nairobi Laptop       23  4030000
        Tablet       27  2492000


In [ ]:
os.makedirs("outputs", exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print("Saved:", check.shape, "| branches:", sorted(check["branch"].unique()),
      "| channels:", check["channel"].unique(), "| bands:", check["price_band"].unique())

Saved: (83, 12) | branches: ['Mombasa', 'Nairobi'] | channels: <StringArray>
['Online']
Length: 1, dtype: str | bands: <StringArray>
['Premium']
Length: 1, dtype: str


**Answer.** There are **83** Premium online orders from Nairobi and Mombasa, worth **KES 10,218,000** in total (8.3% of orders and 14.2% of revenue). They are saved, without the index column, in `outputs/premium_online_coast_capital.csv`.

**Context.** About one order in twelve is in this segment, yet it brings in roughly one shilling in seven, because every order in it is a Laptop or a Tablet. The file is a ready list for a Premium online campaign aimed at the two largest cities.

## 9. Task 8: Excel check (SUMIFS, XLOOKUP, PivotTable)

The workbook `excel/duka_branch_summary.xlsx` has three sheets: **Data** (the 1,000 orders and a `revenue` formula), **Branch Summary** (a `SUMIFS` revenue table, a `COUNTIFS` order count, an `XLOOKUP` region column and a reconciliation check), and **Pivot** (a PivotTable of revenue by branch and channel). The cell below reads Excel's saved results and compares them with pandas.

In [ ]:
import openpyxl

wb = openpyxl.load_workbook("excel/duka_branch_summary.xlsx", data_only=True)

# 1) SUMIFS table vs pandas branch totals
ws = wb["Branch Summary"]
xl_branch = pd.DataFrame([r[:5] for r in ws.iter_rows(min_row=4, max_row=8, values_only=True)],
                         columns=["branch", "region", "xl_revenue", "xl_orders", "xl_rank"]).set_index("branch")
cmp1 = xl_branch.join(branch[["revenue", "orders", "rank_revenue"]])
print("SUMIFS revenue matches pandas :", (cmp1["xl_revenue"] == cmp1["revenue"]).all())
print("COUNTIFS orders match pandas  :", (cmp1["xl_orders"] == cmp1["orders"]).all())
print("Excel rank matches pandas rank:", (cmp1["xl_rank"] == cmp1["rank_revenue"]).all())
print("Excel grand total:", f"{ws['C9'].value:,}", "| pandas total:", f"{df['revenue'].sum():,}")

# 2) PivotTable (branch x channel) vs pandas pivot
rows = [r for r in wb["Pivot"].iter_rows(values_only=True)]
hdr_i = next(i for i, r in enumerate(rows) if r[0] == "branch")
channels = list(rows[hdr_i][1:4])
xl_pivot = pd.DataFrame([r[1:4] for r in rows[hdr_i + 1: hdr_i + 6]], columns=channels,
                        index=[r[0] for r in rows[hdr_i + 1: hdr_i + 6]])
pd_pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
print("PivotTable matches pandas pivot:", xl_pivot.sort_index().astype(float).equals(
      pd_pivot.sort_index()[channels].astype(float)))
pd_pivot

SUMIFS revenue matches pandas : True
COUNTIFS orders match pandas  : True
Excel rank matches pandas rank: True
Excel grand total: 71,792,500 | pandas total: 71,792,500
PivotTable matches pandas pivot: True


channel,In-store,Online,Phone order
branch,,,
Eldoret,4885300,3450700,508200
Kisumu,6468600,3370800,1028700
Mombasa,8039600,4086800,1877300
Nairobi,14808400,7527400,3770300
Nakuru,6767300,4350400,852700


**Result.** All checks print `True`: the Excel `SUMIFS` revenue, `COUNTIFS` orders and rank per branch equal the pandas results, the grand total is KES 71,792,500 in both, and the PivotTable of revenue by branch and channel equals the pandas `pivot_table`.

**Notes.** (1) The region column uses `XLOOKUP` against the small Branch/Region table in columns G:H of the *Branch Summary* sheet. The brief does not define regions, so that mapping is my assumption (Mombasa = Coast; Nakuru and Eldoret = Rift Valley; Kisumu = Lake region). (2) `XLOOKUP` needs Excel 2021 or Microsoft 365 or Google Sheets; older versions show `#NAME?`.